In [107]:
from calendar import weekday

import pandas as pd
import numpy_financial as npf
import openpyxl as xl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime

In [108]:
boe_extract_import = pd.read_excel('/Users/andrewmcclymont/PycharmProjects/PythonProject/data/RMBS_Model/Python_Outputs/BoE_Extract.xlsx')
sonia_fwd_curve = pd.read_excel('/Users/andrewmcclymont/PycharmProjects/PythonProject/data/RMBS_Model/Sonia_fwd_curve.xlsx')

In [109]:
boe_extract_import.head()

,Unnamed: 0,loan_identifier,loan_term,date_of_loan_maturity,maturity_quarter,date_of_loan_maturity_date_converted,remaining_term,current_balance,repayment_method,repayment_method_mapped,payment_due,interest_rate_type,interest_rate_type_mapped,current_interest_rate_index,current_interest_rate_index_mapped,current_interest_rate,current_interest_rate_margin,implied_svr,interest_revision_date_1_date_converted,reversion_term
0,1,9100006288,216,Q3-2039,Q3,2039-09-30,172,40130.30,2,repayment,436.57,1,floating,11,SVR,0.0980,0.0530,0.045,NaT,NaN
1,2,9100008835,360,Q2-2051,Q2,2051-06-30,313,51353.89,2,repayment,359.75,5,fix_to_float,11,SVR,0.0705,0.0330,NaN,2026-01-01,8.0
2,3,9100009832,204,Q2-2038,Q2,2038-06-30,157,438367.46,1,interest_only,1440.00,5,fix_to_float,11,SVR,0.0394,0.0374,NaN,2026-07-01,14.0
3,4,9100011667,300,Q2-2046,Q2,2046-06-30,253,246758.22,1,interest_only,776.16,5,fix_to_float,11,SVR,0.0379,0.0359,NaN,2026-07-01,14.0
4,5,9100011981,300,Q2-2046,Q2,2046-06-30,253,84733.47,1,interest_only,266.88,5,fix_to_float,11,SVR,0.0379,0.0359,NaN,2026-06-01,13.0


In [110]:
sonia_fwd_curve

,Period,Compounded Sonia Forward
0,0,NaN
1,1,0.0439
2,2,0.0436
3,3,0.0433
4,4,0.0427
...,...,...
496,496,0.0550
497,497,0.0550
498,498,0.0550
499,499,0.0550


In [111]:
from src.utils.strings import clean_to_snake_case

#Clean the naming of the columns
sonia_fwd_curve.columns = [clean_to_snake_case(col) for col in sonia_fwd_curve.columns]
sonia_fwd_curve.head()

,period,compounded_sonia_forward
0,0,NaN
1,1,0.0439
2,2,0.0436
3,3,0.0433
4,4,0.0427


In [112]:
# Define model inputs , left as 5 meaning 5%
SVR_input = 4.5

CPR_input = 5
CDR_input = 0.7
Severity_input = 60
recovery_lag = 3

SVR = SVR_input/100
CPR = CPR_input/100
CDR = CDR_input/100
Severity = Severity_input/100

SMM = (1-(1-CPR)**(1/12))
MDR = (1-(1-CDR)**(1/12))

print(SMM)
print(MDR)


0.004265318777560645
0.0005852132739561089


In [113]:
# Choose a loan to look at
loan_id_choice = 3


# update the name of 0
boe_extract_import.rename(columns={'Unnamed: 0': 'loan_id'}, inplace=True)

single_loan_df = boe_extract_import[boe_extract_import['loan_id'] == loan_id_choice]

single_loan_df.reset_index()


,index,loan_id,loan_identifier,loan_term,date_of_loan_maturity,maturity_quarter,date_of_loan_maturity_date_converted,remaining_term,current_balance,repayment_method,...,payment_due,interest_rate_type,interest_rate_type_mapped,current_interest_rate_index,current_interest_rate_index_mapped,current_interest_rate,current_interest_rate_margin,implied_svr,interest_revision_date_1_date_converted,reversion_term
0,2,3,9100009832,204,Q2-2038,Q2,2038-06-30,157,438367.46,1,...,1440.0,5,fix_to_float,11,SVR,0.0394,0.0374,NaN,2026-07-01,14.0


In [114]:
print("\n=== Loan Overview ===")
print(single_loan_df[[
    'loan_id',
    'loan_identifier',
    'current_balance',
    'loan_term',
    'remaining_term'
]].to_string(index=False))

print("\n=== Repayment Info ===")
print(single_loan_df[[
    'repayment_method_mapped',
    'interest_rate_type_mapped',
    'current_interest_rate',
    'current_interest_rate_index_mapped'
]].to_string(index=False))

print("\n=== Model Assumptions ===")
assumption_dict = {
    "SVR_t0": f"{SVR_input}%",
    "CPR": f"{CPR_input}%",
    "CDR": f"{CDR_input}%",
    "Severity": f"{Severity_input}%",
    "SMM": f"{round(SMM*100,3)}%",
    "MDR": f"{round(MDR*100,3)}%"
}

df = pd.DataFrame(
    list(assumption_dict.items()),
    columns=["Assumption", "Value"]
)

print(df.to_string(index=False))


=== Loan Overview ===
 loan_id loan_identifier  current_balance  loan_term  remaining_term
       3      9100009832        438367.46        204             157

=== Repayment Info ===
repayment_method_mapped interest_rate_type_mapped  current_interest_rate current_interest_rate_index_mapped
          interest_only              fix_to_float                 0.0394                                SVR

=== Model Assumptions ===
Assumption  Value
    SVR_t0   4.5%
       CPR     5%
       CDR   0.7%
  Severity    60%
       SMM 0.427%
       MDR 0.059%


In [115]:
#Setting up the model
period_max = 360
max_term = single_loan_df['remaining_term'].max()
period_range = range(0, (max_term + 1))
print(period_range)

range(0, 158)


In [116]:
input_data = {
    "period": period_range,
}

loan_level_contractual = pd.DataFrame(input_data, index=period_range)
loan_level_contractual

,period
0,0
1,1
2,2
3,3
4,4
...,...
153,153
154,154
155,155
156,156


In [117]:
#Merge in the forward curve
loan_level_contractual = loan_level_contractual.merge(sonia_fwd_curve, on='period', how='left')
loan_level_contractual

,period,compounded_sonia_forward
0,0,NaN
1,1,0.0439
2,2,0.0436
3,3,0.0433
4,4,0.0427
...,...,...
153,153,0.0550
154,154,0.0550
155,155,0.0550
156,156,0.0550


In [118]:
single_loan_df.columns.to_list()


['loan_id',
 'loan_identifier',
 'loan_term',
 'date_of_loan_maturity',
 'maturity_quarter',
 'date_of_loan_maturity_date_converted',
 'remaining_term',
 'current_balance',
 'repayment_method',
 'repayment_method_mapped',
 'payment_due',
 'interest_rate_type',
 'interest_rate_type_mapped',
 'current_interest_rate_index',
 'current_interest_rate_index_mapped',
 'current_interest_rate',
 'current_interest_rate_margin',
 'implied_svr',
 'interest_revision_date_1_date_converted',
 'reversion_term']

In [119]:
#set up all the names
order = ['period',
         'compounded_sonia_forward',
         'SVR',
         'margin',
         'interest_rate',
         'BOP',
         'interest_pmt',
         'principal_pmt',
         'EOP'
]

for col in order:
    if col not in loan_level_contractual.columns:
        loan_level_contractual[col] = np.nan  # or None

# Step 2: Reorder columns
loan_level_contractual = loan_level_contractual[order]



In [120]:
#Define the period t0 values
loan_level_contractual.loc[1,'SVR'] = SVR

#Margin
margin_t1 = single_loan_df['current_interest_rate_margin'].iloc[0]
loan_level_contractual.loc[:,'margin'] = margin_t1

#Adding at period 0 to allow for the loop to work
EOP_t0 = single_loan_df['current_balance'].iloc[0]
loan_level_contractual.loc[0,'EOP'] = EOP_t0

loan_level_contractual

,period,compounded_sonia_forward,SVR,margin,interest_rate,BOP,interest_pmt,principal_pmt,EOP
0,0,NaN,NaN,0.0374,NaN,NaN,NaN,NaN,438367.46
1,1,0.0439,0.045,0.0374,NaN,NaN,NaN,NaN,NaN
2,2,0.0436,NaN,0.0374,NaN,NaN,NaN,NaN,NaN
3,3,0.0433,NaN,0.0374,NaN,NaN,NaN,NaN,NaN
4,4,0.0427,NaN,0.0374,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
153,153,0.0550,NaN,0.0374,NaN,NaN,NaN,NaN,NaN
154,154,0.0550,NaN,0.0374,NaN,NaN,NaN,NaN,NaN
155,155,0.0550,NaN,0.0374,NaN,NaN,NaN,NaN,NaN
156,156,0.0550,NaN,0.0374,NaN,NaN,NaN,NaN,NaN


In [121]:
print(single_loan_df['reversion_term'].iloc[0])
print(single_loan_df['interest_rate_type_mapped'].iloc[0])

14.0
fix_to_float


In [122]:
print(max_term)
print(len(period_range))

157
158


In [123]:
# Next need to do the loop of the values:
# using the len-1 as period range is 158, but the loc is 0 to 157, and we start at 1
for i in range(1, len(period_range)-1):
    #Interest Rate of the loan
    loan_level_contractual.loc[i,'BOP'] = loan_level_contractual.loc[i-1,'EOP']
    loan_level_contractual.loc[i+1,'SVR'] = loan_level_contractual.loc[i,'SVR']*(loan_level_contractual.loc[i+1,'compounded_sonia_forward']/loan_level_contractual.loc[i,'compounded_sonia_forward'])

    # handle the last row for the i+1 of the SVR
    loan_level_contractual.loc[len(period_range)-1,'interest_rate'] = (
    loan_level_contractual.loc[len(period_range)-1,'SVR'] + loan_level_contractual.loc[len(period_range)-1,'margin']
    )

    if single_loan_df['interest_rate_type_mapped'].iloc[0] == 'fix_to_float':
        if single_loan_df['reversion_term'].iloc[0] > loan_level_contractual['period'].iloc[i]:
            loan_level_contractual.loc[i,'interest_rate'] = single_loan_df['current_interest_rate'].iloc[0]
            # print(f'{i} and {loan_level_contractual.loc[i,'interest_rate']}')
        else:
            loan_level_contractual.loc[i,'interest_rate'] = (loan_level_contractual.loc[i,'SVR'] + loan_level_contractual.loc[i,'margin'])
    else:
        loan_level_contractual.loc[i,'interest_rate'] = (loan_level_contractual.loc[i,'SVR'] + loan_level_contractual.loc[i,'margin'])



loan_level_contractual

,period,compounded_sonia_forward,SVR,margin,interest_rate,BOP,interest_pmt,principal_pmt,EOP
0,0,NaN,NaN,0.0374,NaN,NaN,NaN,NaN,438367.46
1,1,0.0439,0.045000,0.0374,0.039400,438367.46,NaN,NaN,NaN
2,2,0.0436,0.044692,0.0374,0.039400,NaN,NaN,NaN,NaN
3,3,0.0433,0.044385,0.0374,0.039400,NaN,NaN,NaN,NaN
4,4,0.0427,0.043770,0.0374,0.039400,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
153,153,0.0550,0.056378,0.0374,0.093778,NaN,NaN,NaN,NaN
154,154,0.0550,0.056378,0.0374,0.093778,NaN,NaN,NaN,NaN
155,155,0.0550,0.056378,0.0374,0.093778,NaN,NaN,NaN,NaN
156,156,0.0550,0.056378,0.0374,0.093778,NaN,NaN,NaN,NaN


In [127]:
#Set period 1 - cant us np.where as its a vector approach so all at once no i for the false part
# loan_level_contractual['interest_pmt'] = np.where(loan_level_contractual['period'] == 1,  # condition
#                                     # value if True
#                                     npf.impt(rate = (loan_level_contractual['interest_rate']/12),
#                                              per = 1,
#                                              nper = max_term,
#                                              pv = (-1*loan_level_contractual['BOP'])
#                                             ),
#                                     # value if False
#                                     np.nan)

# Need to set this up for 0
for i in range(1,len(period_range)):
    #set up the BOP and EOP
    loan_level_contractual.loc[i,'BOP'] = loan_level_contractual.loc[i-1,'EOP']

    #For IO mtge, the period is 1, as we do not need to account for the value change of the period
    if single_loan_df['repayment_method_mapped'].iloc[0] == 'interest_only':

        # FIX: last valid row is max_term-1
        loan_level_contractual.loc[i, 'principal_pmt'] = 0
        loan_level_contractual.loc[max_term, 'principal_pmt'] = loan_level_contractual.loc[max_term, 'BOP']

        loan_level_contractual.loc[i,'interest_pmt'] = (
                                                    npf.ipmt(rate=(loan_level_contractual.loc[i,'interest_rate']/12),
                                                            per = 1,
                                                            nper = max_term,
                                                            pv = (loan_level_contractual.loc[i,'BOP']*-1))
                                                    )

    #Princ - for IO we can set this to 0
    else:
        loan_level_contractual.loc[i,'principal_pmt'] = (
                                                    npf.ppmt(rate=(loan_level_contractual.loc[i,'interest_rate']/12),
                                                            per = loan_level_contractual.loc[i,'period'],
                                                            nper = max_term,
                                                            pv = (loan_level_contractual.loc[i,'BOP']*-1))
        )

        loan_level_contractual.loc[i,'interest_pmt'] = (
                                                    npf.ipmt(rate=(loan_level_contractual.loc[i,'interest_rate']/12),
                                                            per = loan_level_contractual.loc[i,'period'],
                                                            nper = max_term,
                                                            pv = (loan_level_contractual.loc[i,'BOP']*-1))
        )


    #Calc EOP
    loan_level_contractual.loc[i,'EOP'] = (loan_level_contractual.loc[i,'BOP'] - loan_level_contractual.loc[i,'principal_pmt'])

    #Add a loan level amort factor
    loan_level_contractual.loc[0,'amort_factor'] = (loan_level_contractual.loc[0,'EOP']/loan_level_contractual.loc[0,'EOP'])
    loan_level_contractual.loc[i,'amort_factor'] = (loan_level_contractual.loc[i,'EOP']/loan_level_contractual.loc[0,'EOP'])


#cover for anything that falls outside of the logic
loan_level_contractual

,period,compounded_sonia_forward,SVR,margin,interest_rate,BOP,interest_pmt,principal_pmt,EOP,amort_factor
0,0,NaN,NaN,0.0374,NaN,NaN,NaN,NaN,438367.46,1.0
1,1,0.0439,0.045000,0.0374,0.039400,438367.46,1439.306494,0.00,438367.46,1.0
2,2,0.0436,0.044692,0.0374,0.039400,438367.46,1439.306494,0.00,438367.46,1.0
3,3,0.0433,0.044385,0.0374,0.039400,438367.46,1439.306494,0.00,438367.46,1.0
4,4,0.0427,0.043770,0.0374,0.039400,438367.46,1439.306494,0.00,438367.46,1.0
...,...,...,...,...,...,...,...,...,...,...
153,153,0.0550,0.056378,0.0374,0.093778,438367.46,3425.773465,0.00,438367.46,1.0
154,154,0.0550,0.056378,0.0374,0.093778,438367.46,3425.773465,0.00,438367.46,1.0
155,155,0.0550,0.056378,0.0374,0.093778,438367.46,3425.773465,0.00,438367.46,1.0
156,156,0.0550,0.056378,0.0374,0.093778,438367.46,3425.773465,0.00,438367.46,1.0


**Now we add the CDR/CPR as a Survival Factor done so far**

To add:
1) Intex Survival Factor analysis
2) flat value to see the diff to (1)
3) Vector
4) RA approach

# Define model inputs , left as 5 meaning 5%
SVR_input = 4.5

CPR_input = 5
CDR_input = 0.7
Severity_input = 60
recovery_lag = 3

SVR = SVR_input/100
CPR = CPR_input/100
CDR = CDR_input/100
Severity = Severity_input/100

SMM = (1-(1-CPR)**(1/12))
MDR = (1-(1-CDR)**(1/12))

print(SMM)
print(MDR)

In [125]:
# loan_level_contractual_surv_factor = loan_level_contractual
#
# #New columns made in the order I want
# loan_level_contractual_surv_factor['sched_payment'] = None
# loan_level_contractual_surv_factor['survival_factor'] = None
# loan_level_contractual_surv_factor['surviving_scheduled_payment'] = None
# loan_level_contractual_surv_factor['new_interest'] = None
# loan_level_contractual_surv_factor['new_principal'] = None
# loan_level_contractual_surv_factor['survival_adjusted_balance'] = None
# loan_level_contractual_surv_factor['default_amount'] = None
# loan_level_contractual_surv_factor['prepayment_actual_scheduled'] = None
# loan_level_contractual_surv_factor['BOP_adj'] = None
# loan_level_contractual_surv_factor['EOP_adj'] = None
#
# # Need to add in the SMM and MDR cols if we want to change the logic here <- more of a futre part for a function
#
# for i in range(1,len(period_range)):
#     #set up the BOP and EOP
#     loan_level_contractual_surv_factor.loc[i,'BOP'] = loan_level_contractual_surv_factor.loc[i-1,'EOP']
#
#     #For IO mtge, the period is 1, as we do not need to account for the value change of the period
#     if single_loan_df['repayment_method_mapped'].iloc[0] == 'interest_only':
#
#         # FIX: last valid row is max_term-1
#         loan_level_contractual_surv_factor.loc[i, 'principal_pmt'] = 0
#         loan_level_contractual_surv_factor.loc[max_term, 'principal_pmt'] = loan_level_contractual_surv_factor.loc[max_term, 'BOP']
#
#         loan_level_contractual_surv_factor.loc[i,'interest_pmt'] = (
#                                                     npf.ipmt(rate=(loan_level_contractual_surv_factor.loc[i,'interest_rate']/12),
#                                                             per = 1,
#                                                             nper = max_term,
#                                                             pv = (loan_level_contractual_surv_factor.loc[i,'BOP']*-1))
#                                                     )
#
#     #Princ - for IO we can set this to 0
#     else:
#         loan_level_contractual_surv_factor.loc[i,'principal_pmt'] = (
#                                                     npf.ppmt(rate=(loan_level_contractual_surv_factor.loc[i,'interest_rate']/12),
#                                                             per = loan_level_contractual_surv_factor.loc[i,'period'],
#                                                             nper = max_term,
#                                                             pv = (loan_level_contractual_surv_factor.loc[i,'BOP']*-1))
#         )
#
#         loan_level_contractual_surv_factor.loc[i,'interest_pmt'] = (
#                                                     npf.ipmt(rate=(loan_level_contractual_surv_factor.loc[i,'interest_rate']/12),
#                                                             per = loan_level_contractual_surv_factor.loc[i,'period'],
#                                                             nper = max_term,
#                                                             pv = (loan_level_contractual_surv_factor.loc[i,'BOP']*-1))
#         )
#
#
#     #Calc EOP
#     loan_level_contractual_surv_factor.loc[i,'EOP'] = (loan_level_contractual_surv_factor.loc[i,'BOP'] - loan_level_contractual_surv_factor.loc[i,'principal_pmt'])
#
# #cover for anything that falls outside of the logic
# loan_level_contractual_surv_factor

In [140]:
# Redo doing just the info on MDR and CDR <-- make it so we can pass any curve in
# Using an amort factor of the none adjusted so the contractual

loan_level_contractual_scen = loan_level_contractual.copy()

# Convert MDR + SMM to vectors if needed
if np.isscalar(MDR):
    MDR_vec = np.full(len(period_range), MDR)
else:
    MDR_vec = np.asarray(MDR)

if np.isscalar(SMM):
    SMM_vec = np.full(len(period_range), SMM)
else:
    SMM_vec = np.asarray(SMM)


BOP_adjCDR = np.zeros(len(period_range))
BOP_adjCDR_CPR = np.zeros(len(period_range))
amort_delta = np.zeros(len(period_range))

#Interest Rate of the loan
loan_level_contractual_scen.loc[1,'BOP'] = loan_level_contractual_scen.loc[0,'EOP']

for i in range(1, len(period_range)-1):

    #Period, Rate, SVR, Margin, Interest
    loan_level_contractual_scen.loc[i+1,'SVR'] = (
        loan_level_contractual_scen.loc[i,'SVR'] *
        (loan_level_contractual_scen.loc[i+1,'compounded_sonia_forward'] /
         loan_level_contractual_scen.loc[i,'compounded_sonia_forward'])
    )

    # handle the last row for the i+1 of the SVR
    loan_level_contractual_scen.loc[len(period_range)-1,'interest_rate'] = (
        loan_level_contractual_scen.loc[len(period_range)-1,'SVR'] +
        loan_level_contractual_scen.loc[len(period_range)-1,'margin']
    )

    if single_loan_df['interest_rate_type_mapped'].iloc[0] == 'fix_to_float':
        if single_loan_df['reversion_term'].iloc[0] > loan_level_contractual_scen['period'].iloc[i]:
            loan_level_contractual_scen.loc[i,'interest_rate'] = single_loan_df['current_interest_rate'].iloc[0]
        else:
            loan_level_contractual_scen.loc[i,'interest_rate'] = (
                loan_level_contractual_scen.loc[i,'SVR'] + loan_level_contractual_scen.loc[i,'margin']
            )
    else:
        loan_level_contractual_scen.loc[i,'interest_rate'] = (
            loan_level_contractual_scen.loc[i,'SVR'] + loan_level_contractual_scen.loc[i,'margin']
        )

    # ----------------------------------------------------------------------------------------------

    loan_level_contractual_scen.loc[i,'BOP'] = loan_level_contractual_scen.loc[i-1,'EOP']

    #Default rate <- add here logic for the vector / timing would be of 1,BOP
    loan_level_contractual_scen.loc[i, 'deafult_rate'] = MDR_vec[i]
    loan_level_contractual_scen.loc[i, 'default_amt'] = MDR_vec[i] * loan_level_contractual_scen.loc[i, 'BOP']

    BOP_adjCDR[i] = loan_level_contractual_scen.loc[i,'BOP'] - loan_level_contractual_scen.loc[i,'default_amt']

    #Prepay rate <- add here logic for the vector / timing would be of 1,BOP
    loan_level_contractual_scen.loc[i, 'prepay_rate'] = SMM_vec[i]
    loan_level_contractual_scen.loc[i, 'prepay_amt'] = SMM_vec[i] * BOP_adjCDR[i]

    BOP_adjCDR_CPR[i] = (
        loan_level_contractual_scen.loc[i,'BOP']
        - loan_level_contractual_scen.loc[i,'default_amt']
        - loan_level_contractual_scen.loc[i,'prepay_amt']
    )

    #Calculating the int and prin off the factor now and adjusted for defautlt
    amort_delta[i] = (1 - (loan_level_contractual_scen.loc[i,'amort_factor'] /
                           loan_level_contractual_scen.loc[i-1,'amort_factor']
                           )
                      )
    loan_level_contractual_scen.loc[i,'actual_principal_pmt'] = amort_delta[i] * BOP_adjCDR_CPR[i]

    loan_level_contractual_scen.loc[i,'actual_interest_pmt'] = (loan_level_contractual_scen.loc[i,'interest_rate'] / 12
                                                                ) * BOP_adjCDR[i]

    #Calc EOP
    loan_level_contractual_scen.loc[i,'EOP'] = (
        loan_level_contractual_scen.loc[i,'BOP']
        - loan_level_contractual_scen.loc[i,'default_amt']
        - loan_level_contractual_scen.loc[i,'prepay_amt']
        - loan_level_contractual_scen.loc[i,'actual_principal_pmt']
    )

    # ------------------------------------------------------------------------
    # Set up the recovery
    # recovery_lag = 3 - so need to lag it
    if i < recovery_lag:
        loan_level_contractual_scen.loc[i,'recovery_pmt'] = 0
    else:
        loan_level_contractual_scen.loc[i,'recovery_pmt'] = (
            loan_level_contractual_scen.loc[i-recovery_lag,'default_amt'] * (1-Severity)
        )

    # ------------------------------------------------------------------------
    loan_level_contractual_scen.loc[i,'total_prin_pmt'] = (
        loan_level_contractual_scen.loc[i,'actual_principal_pmt'] +
        loan_level_contractual_scen.loc[i,'prepay_amt']
    )

    loan_level_contractual_scen.loc[i,'total_int_pmt'] = (
        loan_level_contractual_scen.loc[i,'actual_interest_pmt'] +
        loan_level_contractual_scen.loc[i,'recovery_pmt']
    )

# -------------------------------------

#set up all the names
order2 = [
    'period',
    'compounded_sonia_forward',
    'SVR',
    'margin',
    'interest_rate',
    'BOP',
    'deafult_rate',
    'default_amt',
    'amort_factor',
    'prepay_rate',
    'prepay_amt',
    'actual_principal_pmt',
    'actual_interest_pmt',
    'recovery_pmt',
    'EOP',
    'total_prin_pmt',
    'total_int_pmt'
]

for col in order2:
    if col not in loan_level_contractual_scen.columns:
        loan_level_contractual_scen[col] = np.nan

loan_level_contractual_scen = loan_level_contractual_scen[order2]

loan_level_contractual_scen


,period,compounded_sonia_forward,SVR,margin,interest_rate,BOP,deafult_rate,default_amt,amort_factor,prepay_rate,prepay_amt,actual_principal_pmt,actual_interest_pmt,recovery_pmt,EOP,total_prin_pmt,total_int_pmt
0,0,NaN,NaN,0.0374,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,438367.460000,NaN,NaN
1,1,0.0439,0.045000,0.0374,0.039400,438367.460000,0.000585,256.538456,1.0,0.004265,1868.682740,0.0,1438.464192,0.000000,436242.238803,1868.682740,1438.464192
2,2,0.0436,0.044692,0.0374,0.039400,436242.238803,0.000585,255.294749,1.0,0.004265,1859.623299,0.0,1431.490466,0.000000,434127.320755,1859.623299,1431.490466
3,3,0.0433,0.044385,0.0374,0.039400,434127.320755,0.000585,254.057071,1.0,0.004265,1850.607779,0.0,1424.550549,NaN,432022.655906,1850.607779,NaN
4,4,0.0427,0.043770,0.0374,0.039400,432022.655906,0.000585,252.825393,1.0,0.004265,1841.635966,0.0,1417.644277,102.615383,429928.194547,1841.635966,1520.259659
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153,153,0.0550,0.056378,0.0374,0.093778,209424.646130,0.000585,122.558083,1.0,0.004265,892.740126,0.0,1635.663239,49.743202,208409.347921,892.740126,1685.406441
154,154,0.0550,0.056378,0.0374,0.093778,208409.347921,0.000585,121.963917,1.0,0.004265,888.412090,0.0,1627.733485,49.502045,207398.971914,888.412090,1677.235530
155,155,0.0550,0.056378,0.0374,0.093778,207398.971914,0.000585,121.372631,1.0,0.004265,884.105036,0.0,1619.842174,49.262057,206393.494246,884.105036,1669.104232
156,156,0.0550,0.056378,0.0374,0.093778,206393.494246,0.000585,120.784212,1.0,0.004265,879.818863,0.0,1611.989121,49.023233,205392.891170,879.818863,1661.012354


**Adding in some graphs to visualise the cashflow run**